In [64]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
import warnings
warnings.filterwarnings('ignore')

In [65]:

df_decl = pd.read_csv(r"../dataset/web_disaster_declarations.csv")
df_summaries = pd.read_csv(r"../dataset/disaster_summaries.csv")
df_funding = pd.read_csv(r"../dataset/pa_funding_details.csv")



In [92]:
print(f"WEB DISASTER DECLARATIONS. \nShape of Dataset: {df_decl.shape}")
df_decl.head(4)

WEB DISASTER DECLARATIONS. 
Shape of Dataset: (5168, 12)


,disasterNumber,declarationType,incidentType,stateCode,region,declarationDate,declarationRequestDate,incidentBeginDate,incidentEndDate,iaProgramDeclared,paProgramDeclared,hmProgramDeclared
0,5243,Fire Management,Fire,OR,10,2018-06-22T00:00:00.000Z,2018-06-21T00:00:00.000Z,2018-06-21T00:00:00.000Z,2018-06-25T00:00:00.000Z,0.0,1.0,1.0
1,5554,Fire Management,Fire,SC,4,2025-03-07T00:00:00.000Z,2025-03-07T00:00:00.000Z,2025-03-01T00:00:00.000Z,NaN,0.0,1.0,1.0
2,4859,Major Disaster,Severe Storm,AK,10,2025-01-15T00:00:00.000Z,2024-12-16T00:00:00.000Z,2024-10-20T00:00:00.000Z,2024-10-23T00:00:00.000Z,0.0,1.0,1.0
3,4856,Major Disaster,Fire,CA,9,2025-01-08T00:00:00.000Z,2025-01-08T00:00:00.000Z,2025-01-07T00:00:00.000Z,2025-01-31T00:00:00.000Z,0.0,1.0,1.0


In [67]:
df_decl['iaProgramDeclared'].value_counts()

iaProgramDeclared
0.0    3513
1.0    1404
Name: count, dtype: int64

In [69]:
df_decl['disasterNumber'].nunique()

5168

In [70]:
# 2. Check for disasters in Summaries that ARE NOT in Declarations
decl_ids = set(df_decl['disasterNumber'].unique())
summaries_ids = set(df_summaries['disasterNumber'].unique())
orphans = decl_ids - summaries_ids
len(orphans)

1259

In [93]:
print(f"DISASTER SUMMARIES. \nShape of Dataset: {df_summaries.shape}")
df_summaries.head()

DISASTER SUMMARIES. 
Shape of Dataset: (3909, 7)


,disasterNumber,totalObligatedAmountPa,totalObligatedAmountCatAb,totalObligatedAmountCatC2g,totalAmountIhpApproved,totalObligatedAmountHmgp,totalNumberIaApproved
0,3601,NaN,NaN,NaN,NaN,NaN,NaN
1,3602,NaN,NaN,NaN,NaN,NaN,NaN
2,1981,2.243558e+08,94967791.86,1.197739e+08,95754618.67,71549737.00,7469.0
3,1802,1.881275e+07,13423088.38,4.869890e+06,NaN,2710679.00,NaN
4,4317,7.731984e+07,11526563.99,6.437977e+07,12527583.31,17801245.73,1978.0


In [73]:
df_summaries.isna().sum()

disasterNumber                   0
totalObligatedAmountPa         924
totalObligatedAmountCatAb     1185
totalObligatedAmountCatC2g    2361
totalAmountIhpApproved        3308
totalObligatedAmountHmgp      1098
totalNumberIaApproved         3308
dtype: int64

In [74]:
df_summaries['disasterNumber'].nunique()

3909

In [94]:
print(f"PUBLIC ASSISTANCE FUNDING. \nShape of Dataset: {df_funding.shape}")
df_funding.head(4)

PUBLIC ASSISTANCE FUNDING. 
Shape of Dataset: (810695, 4)


,disasterNumber,projectAmount,federalShareObligated,totalObligated
0,1239,100000.00,75000.00,80340.00
1,1239,19685.50,14764.13,15461.00
2,1239,26111.00,19583.25,20507.58
3,1603,18602329.78,18602329.78,18788818.15


In [19]:
df_funding['disasterNumber'].nunique()

1766

In [82]:
df_funding_flattened = df_funding.groupby('disasterNumber').aggregate({
    'federalShareObligated': 'sum',
    'totalObligated': 'sum'
}).reset_index()
df_funding_flattened.head()

,disasterNumber,federalShareObligated,totalObligated
0,1239,7950369.84,8224551.92
1,1257,32229051.37,33271088.84
2,1260,9485830.00,9863594.04
3,1261,5498755.69,5735788.22
4,1262,14941466.64,15374967.08


In [83]:
final_df = df_summaries.merge(df_decl, on='disasterNumber', how='left').merge(df_funding_flattened, on='disasterNumber', how='left')
final_df.head()

,disasterNumber,totalObligatedAmountPa,totalObligatedAmountCatAb,totalObligatedAmountCatC2g,totalAmountIhpApproved,totalObligatedAmountHmgp,totalNumberIaApproved,declarationType,incidentType,stateCode,region,declarationDate,declarationRequestDate,incidentBeginDate,incidentEndDate,iaProgramDeclared,paProgramDeclared,hmProgramDeclared,federalShareObligated,totalObligated
0,3601,NaN,NaN,NaN,NaN,NaN,NaN,Emergency,Tropical Storm,GU,9,2023-10-08T00:00:00.000Z,2023-10-08T00:00:00.000Z,2023-10-08T00:00:00.000Z,2023-10-16T00:00:00.000Z,0.0,1.0,0.0,NaN,NaN
1,3602,NaN,NaN,NaN,NaN,NaN,NaN,Emergency,Tropical Storm,MP,9,2023-10-08T00:00:00.000Z,2023-10-08T00:00:00.000Z,2023-10-09T00:00:00.000Z,2023-10-16T00:00:00.000Z,0.0,1.0,0.0,NaN,NaN
2,1981,2.243558e+08,94967791.86,1.197739e+08,95754618.67,71549737.00,7469.0,Major Disaster,Flood,ND,8,2011-05-10T00:00:00.000Z,2011-05-06T00:00:00.000Z,2011-02-14T00:00:00.000Z,2011-07-20T00:00:00.000Z,0.0,1.0,1.0,2.243454e+08,2.243454e+08
3,1802,1.881275e+07,13423088.38,4.869890e+06,NaN,2710679.00,NaN,Major Disaster,Severe Storm,KY,4,2008-10-09T00:00:00.000Z,2008-09-26T00:00:00.000Z,2008-09-12T00:00:00.000Z,2008-09-14T00:00:00.000Z,0.0,1.0,1.0,1.881275e+07,1.881275e+07
4,4317,7.731984e+07,11526563.99,6.437977e+07,12527583.31,17801245.73,1978.0,Major Disaster,Flood,MO,7,2017-06-02T00:00:00.000Z,2017-05-24T00:00:00.000Z,2017-04-28T00:00:00.000Z,2017-05-11T00:00:00.000Z,0.0,1.0,1.0,7.732569e+07,7.732569e+07


In [90]:
final_df[final_df['totalObligated'].notna()][['disasterNumber', 'totalObligatedAmountPa', 'totalObligated']]

,disasterNumber,totalObligatedAmountPa,totalObligated
2,1981,2.243558e+08,2.243454e+08
3,1802,1.881275e+07,1.881275e+07
4,4317,7.731984e+07,7.732569e+07
5,1292,2.981058e+08,2.981387e+08
7,3163,5.380816e+06,5.380814e+06
...,...,...,...
3864,3219,1.270610e+07,1.270610e+07
3865,3250,2.541600e+06,2.541600e+06
3875,3229,1.036862e+06,1.036773e+06
3876,4456,6.543056e+06,6.543056e+06


In [85]:
final_df.isna().sum()

disasterNumber                   0
totalObligatedAmountPa         924
totalObligatedAmountCatAb     1185
totalObligatedAmountCatC2g    2361
totalAmountIhpApproved        3308
totalObligatedAmountHmgp      1098
totalNumberIaApproved         3308
declarationType                  0
incidentType                     0
stateCode                        0
region                           0
declarationDate                  0
declarationRequestDate           1
incidentBeginDate                0
incidentEndDate                296
iaProgramDeclared              251
paProgramDeclared              251
hmProgramDeclared              251
federalShareObligated         2144
totalObligated                2144
dtype: int64